# EFHM return periods — how flood depth grows with rarity

A return period expresses how rare a flood is: a 1-in-10-year event is common and
shallow, a 1-in-500-year event is rare and deep. This notebook fetches three
return periods for the same area and compares their depths — one windowed read
per period.

In [ ]:
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from pyramids.dataset import Dataset, GeoReference

from earthlens.core import EarthLens

out = Path(tempfile.mkdtemp(prefix="efhm-rp-"))
lat_lim, lon_lim = [51.7, 52.0], [4.6, 5.1]
periods = [10, 100, 500]
paths = EarthLens(
    data_source="jrc-flood",
    lat_lim=lat_lim,
    lon_lim=lon_lim,
    return_periods=periods,
    path=out,
).download()
[p.name for p in paths]

## Side-by-side depth maps

In [ ]:
def _depth(path):
    """Return the blanked depth grid as a Dataset plus its raw array."""
    source = Dataset.read_file(path)
    a = source.read_array().astype("float64")
    a[a <= -9999] = np.nan
    grid = Dataset.from_array(
        a,
        no_data_value=np.nan,
        geo_ref=GeoReference(geo=source.geotransform, epsg=source.epsg),
    )
    return grid, a


loaded = [_depth(p) for p in paths]
grids = [g for g, _ in loaded]
depths = [a for _, a in loaded]
vmax = np.nanmax([np.nanmax(d) for d in depths])

# matplotlib lays the row out; pyramids draws each georeferenced panel into it.
fig, axes = plt.subplots(1, len(periods), figsize=(5 * len(periods), 4.5))
for ax, rp, grid in zip(axes, periods, grids):
    glyph = grid.plot(
        fig=fig,
        ax=ax,
        cmap="Blues",
        vmin=0,
        vmax=vmax,
        title=f"RP{rp} (1-in-{rp}-year)",
    )
    glyph.cbar.set_label("depth (m)")

## Flooded area and depth grow with the return period

In [ ]:
for rp, d in zip(periods, depths):
    flooded = np.isfinite(d).sum()
    print(
        f"RP{rp:>3}: {flooded:>7,} flooded cells | mean {np.nanmean(d):.2f} m | max {np.nanmax(d):.2f} m"
    )

## Takeaway

Rarer floods inundate more cells and reach greater depths. Selecting the right
return period lets you match a hazard layer to a design standard (e.g. RP100 for
many flood defences, RP500 for critical infrastructure).